# A lab-data agent: measuring *g* on an unknown planet

**What is an AI agent?** A language model (LLM) placed in a **loop** with **tools**: at each step the LLM *decides* what to do next (which tool, with which arguments), the code *executes* it, and the result is fed back to the LLM, until the task is done. The LLM reasons; the tools act and compute.

**Why this experiment?** Agents are only useful when the answer **cannot** come from the model's memory. Here:
- the measurements come from a pendulum on an **unknown planet**: *g* is drawn at random and hidden, so no model can know it;
- the data files are **messy** (a length in cm, a large-amplitude run, a corrupted file), so the agent must **look at the data and decide** what to trust, not follow a fixed script.

**Physics:** a simple pendulum of length $L$ has period $T = 2\pi\sqrt{L/g}$ for small oscillations, so $T^2 = \frac{4\pi^2}{g}\,L$: fitting $T^2$ against $L$ gives $g$.

**Plan of the notebook**
1. Generate the measurements (hidden *g*) and save them as CSV files.
2. Build the agent: an LLM call, the agent loop, and the analysis tools.
3. Run the agent, compare with the same LLM **without tools**, then reveal the true *g*.

**Saved run:** the agent found *g* within **0.1%** of the hidden value and caught all three data problems (see *Results* at the end).

**Requirements:** Google Colab with a **T4 GPU** (*Runtime → Change runtime type*); add a first cell `!pip install -q transformers accelerate`. The model (~5 GB) downloads on first run; the agent then takes ~1 minute. On CPU it can take hours.

## Setup

- The LLM is **MiniCPM** (`openbmb/MiniCPM5-2B`): a small open-source model running locally with 🤗 `transformers`, no API key needed.
- NumPy/SciPy simulate the pendulum and analyse the data.

In [1]:
import os, json, time, traceback
from dataclasses import dataclass, field
from typing import List, Callable, Dict, Any

import numpy as np
from scipy.integrate import solve_ivp      # ODE integrator (physics tools)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# Load MiniCPM (~5 GB, downloaded on first run, then cached)
model_id = "openbmb/MiniCPM5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

Model loaded successfully!


## Step 1 · Generate the measurements and save them

- *g* is drawn at random between 3 and 20 m/s² and stored in `_G_TRUE`, **never printed** until the end.
- Each run: damped nonlinear pendulum $\ddot\theta = -\frac{g}{L}\sin\theta - \gamma\dot\theta$, sampled at 50 Hz for 30 s, with 0.2° Gaussian noise (like a real angle sensor).
- One CSV per run in `lab_data/` (`time_s, angle_deg`); the length appears **only in the file name**.

| File | Length | Amplitude | Problem |
|---|---|---|---|
| `run01_L=0.25m.csv` | 0.25 m | 8° | – |
| `run02_L=0.50m.csv` | 0.50 m | 7° | – |
| `run03_L=75cm.csv` | 0.75 m | 8° | length in **cm** |
| `run04_L=1.00m.csv` | 1.00 m | 6° | – |
| `run05_L=1.25m.csv` | 1.25 m | 50° | **large amplitude**: $T = 2\pi\sqrt{L/g}$ no longer valid |
| `run06_L=1.50m.csv` | 1.50 m | 8° | **corrupted**: stops after 2.4 s, `ERR` values |

The agent is **not** told about these problems: it has to discover them.

In [2]:
def generate_lab_data(folder="lab_data", seed=None):
    rng = np.random.default_rng(seed)
    g_true = float(rng.uniform(3.0, 20.0))          # hidden: an unknown planet
    gamma, fs, duration, noise_deg = 0.05, 50, 30.0, 0.2
    runs = [  # (file name, length [m], amplitude [deg], problem)
        ("run01_L=0.25m.csv", 0.25, 8,  None),
        ("run02_L=0.50m.csv", 0.50, 7,  None),
        ("run03_L=75cm.csv",  0.75, 8,  "length in cm"),
        ("run04_L=1.00m.csv", 1.00, 6,  None),
        ("run05_L=1.25m.csv", 1.25, 50, "large amplitude"),
        ("run06_L=1.50m.csv", 1.50, 8,  "corrupted"),
    ]
    os.makedirs(folder, exist_ok=True)
    for name, L, amp, problem in runs:
        rhs = lambda t, y: [y[1], -(g_true / L) * np.sin(y[0]) - gamma * y[1]]
        t = np.arange(0, duration, 1 / fs)
        sol = solve_ivp(rhs, [0, duration], [np.radians(amp), 0.0], t_eval=t, rtol=1e-9, atol=1e-9)
        angle = np.degrees(sol.y[0]) + rng.normal(0, noise_deg, t.size)
        rows = [f"{ti:.3f},{ai:.3f}" for ti, ai in zip(t, angle)]
        if problem == "corrupted":
            rows = rows[:120]                                   # recording stopped after 2.4 s
            for k in rng.choice(len(rows), 30, replace=False):  # and some values are garbage
                rows[k] = rows[k].split(",")[0] + ",ERR"
        with open(os.path.join(folder, name), "w") as f:
            f.write("time_s,angle_deg\n" + "\n".join(rows) + "\n")
    return g_true



import shutil
shutil.rmtree("lab_data", ignore_errors=True)
_G_TRUE = generate_lab_data("lab_data")        # hidden until the end
print(sorted(os.listdir("lab_data")))
with open("lab_data/run01_L=0.25m.csv") as f:
    print(f.read()[:120])   # first lines of one file

['run01_L=0.25m.csv', 'run02_L=0.50m.csv', 'run03_L=75cm.csv', 'run04_L=1.00m.csv', 'run05_L=1.25m.csv', 'run06_L=1.50m.csv']
time_s,angle_deg
0.000,8.039
0.020,7.948
0.040,7.971
0.060,7.002
0.080,6.750
0.100,6.365
0.120,5.858
0.140,5.005
0.160,4


## Step 2 · How the agent works

### The agent loop

```
┌─► 1. Construct prompt   (instructions + tools + memory of what happened so far)
│   2. Generate response  (the LLM chooses the next tool and its arguments)
│   3. Parse response     (extract {"tool": ..., "args": ...} from the text)
│   4. Execute action     (run the Python function)
│   5. Update memory      (store the decision and the result)
└── 6. Continue?          (stop when the agent calls `terminate`, or after max iterations)
```

### GAME: the four pieces that define an agent

| | Component | Here | Class |
|---|---|---|---|
| **G** | Goals: what to achieve and how | analyse the files, exclude bad runs, fit *g* | `Goal` |
| **A** | Actions: what the agent can do | `list_files`, `inspect_file`, `measure_period`, `fit_g`, `terminate` | `Action`, `ActionRegistry` |
| **M** | Memory: what has happened | the conversation: task, decisions, results | `Memory` |
| **E** | Environment: where actions run | executes the tools, returns results or errors | `Environment` |

The loop is written once; a different agent only needs different goals and actions. This small framework is adapted from the Coursera course *AI Agents and Agentic AI with Python & Generative AI* (Vanderbilt University); see [`course/`](../course/).

### The LLM call

MiniCPM has no built-in *function calling* (the API feature of large models that returns tool calls in a structured way), so `generate_response` emulates it:
1. the tool descriptions (JSON schemas) are written into the system message, with the required format ```` ```action {"tool": ..., "args": ...} ``` ````;
2. MiniCPM generates text (it may think step by step first);
3. a **tolerant parser** extracts the tool call (also accepts `<action>` tags or ```` ```json ````, common slips of small models);
4. if no valid call is found, the model gets **one retry** with a message saying what was wrong.

It returns the call as `'{"tool": ..., "args": ...}'`, or plain text if the model did not call a tool.

In [4]:
@dataclass
class Prompt:
    messages: List[Dict] = field(default_factory=list)   # goals + memory, as chat messages
    tools: List[Dict] = field(default_factory=list)      # actions, as JSON schemas
    metadata: dict = field(default_factory=dict)         # default_factory avoids a shared mutable default


def minicpm_chat(messages: List[Dict], max_new_tokens: int = 1024) -> str:
    """Send chat messages to MiniCPM and return its reply as text (same helper as notebooks 0-2)."""
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # skip MiniCPM's hidden reasoning step
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():             # no gradients: we are not training
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Decode only the new tokens (drop the prompt)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def tools_to_instructions(tools: List[Dict]) -> str:
    """Describe the tools in plain text, since MiniCPM has no function-calling API."""
    tool_list = [
        {
            "name": t["function"]["name"],
            "description": t["function"]["description"],
            "parameters": t["function"]["parameters"],
        }
        for t in tools
    ]
    return (
        "You can use the following tools:\n"
        "```json\n" + json.dumps(tool_list, indent=2) + "\n```\n\n"
        "Every response MUST call exactly one tool. First think step by step, "
        "then write the tool call as a Markdown code block in exactly this format "
        "(a code block, not XML tags):\n"
        "```action\n"
        '{"tool": "<tool name>", "args": {<arguments>}}\n'
        "```"
    )


def extract_tool_call(text: str, tool_names: List[str]):
    """Find a tool call in the model's text. Returns {"tool": ..., "args": ...} or None.

    Tolerant: tries ```action, <action>...</action>, ```json and finally the whole text.
    """
    candidates = []
    if "```action" in text:
        candidates.append(text.split("```action", 1)[1].split("```", 1)[0])
    if "<action>" in text:
        candidates.append(text.split("<action>", 1)[1].split("</action>", 1)[0])
    if "```json" in text:
        candidates.append(text.split("```json", 1)[1].split("```", 1)[0])
    candidates.append(text)

    for candidate in candidates:
        try:
            data = json.loads(candidate.strip())
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict):
            name = data.get("tool") or data.get("tool_name") or data.get("name")
            if name in tool_names:
                return {"tool": name, "args": data.get("args") or data.get("arguments") or {}}
    return None


def generate_response(prompt: Prompt, max_retries: int = 1) -> str:
    """Call the LLM. Returns a tool call as a JSON string, or plain text.

    Same output as the course's GPT-4o version:
        '{"tool": "<name>", "args": {...}}'  if the model called a tool,
        the model's text                      otherwise.
    """
    messages = [dict(m) for m in prompt.messages]          # copy: don't modify the prompt

    # No tools: just chat
    if not prompt.tools:
        return minicpm_chat(messages)

    # 1. Tools -> text instructions, appended to the system message
    instructions = tools_to_instructions(prompt.tools)
    if messages and messages[0]["role"] == "system":
        messages[0]["content"] += "\n\n" + instructions
    else:
        messages.insert(0, {"role": "system", "content": instructions})

    tool_names = [t["function"]["name"] for t in prompt.tools]

    for attempt in range(max_retries + 1):
        # 2. Generate
        text = minicpm_chat(messages)

        # 3. Extract the tool call
        call = extract_tool_call(text, tool_names)
        if call is not None:
            return json.dumps(call)

        # 4. Retry with an actionable error message
        messages += [
            {"role": "assistant", "content": text},
            {"role": "user", "content":
                "Your answer did not contain a valid tool call. Available tools: "
                + ", ".join(tool_names) + ". Reply with one tool call in this format:\n"
                '```action\n{"tool": "<tool name>", "args": {...}}\n```'},
        ]

    return text   # still no tool call: return the text (the agent language will handle it)

### G · A · M · E classes

- `Goal`: a named instruction with a priority (all goals become the system message).
- `Action`: a tool = Python function + description + JSON schema of its arguments + `terminal` flag (`True` ends the loop).
- `ActionRegistry`: finds the `Action` from the name the LLM writes.
- `Memory`: list of items `{"type": "user" | "assistant" | "environment", "content": ...}`.
- `Environment`: runs an action; errors are **returned** to the agent (not raised), so it can react to them.

In [5]:
@dataclass(frozen=True)          # frozen: a goal is immutable once created
class Goal:
    priority: int                # order of importance (used to sort/format goals)
    name: str                    # short label
    description: str             # what to achieve and how


class Action:
    """One tool the agent can use: its Python function + the description the LLM sees."""

    def __init__(self,
                 name: str,               # name the LLM uses to call it
                 function: Callable,      # Python function that does the work
                 description: str,        # what it does (shown to the LLM)
                 parameters: Dict,        # JSON schema of the arguments (shown to the LLM)
                 terminal: bool = False): # True -> running it ends the agent loop
        self.name = name
        self.function = function
        self.description = description
        self.terminal = terminal
        self.parameters = parameters

    def execute(self, **args) -> Any:
        """Execute the action's function"""
        return self.function(**args)


class ActionRegistry:
    """Lookup table name -> Action."""

    def __init__(self):
        self.actions = {}

    def register(self, action: Action):
        self.actions[action.name] = action

    def get_action(self, name: str) -> [Action, None]:
        return self.actions.get(name, None)          # None if the LLM invents a tool name

    def get_actions(self) -> List[Action]:
        """Get all registered actions"""
        return list(self.actions.values())


class Memory:
    """The agent's history: a list of dicts like {"type": "assistant", "content": "..."}."""

    def __init__(self):
        self.items = []  # basic conversation history

    def add_memory(self, memory: dict):
        """Add memory to working memory"""
        self.items.append(memory)

    def get_memories(self, limit: int = None) -> List[Dict]:
        """Get formatted conversation history for prompt"""
        # Note: items[:limit] returns the FIRST `limit` items. To get the most
        # recent ones (as the course text describes), it would be items[-limit:].
        # With limit=None (the default used here) both return everything.
        return self.items[:limit]

    def copy_without_system_memories(self):
        """Return a copy of the memory without system memories"""
        filtered_items = [m for m in self.items if m["type"] != "system"]
        memory = Memory()
        memory.items = filtered_items
        return memory


class Environment:
    """Runs actions and wraps their results (or errors) in a standard dict."""

    def execute_action(self, action: Action, args: dict) -> dict:
        """Execute an action and return the result."""
        try:
            result = action.execute(**args)
            return self.format_result(result)
        except Exception as e:
            # The error goes back to the agent instead of crashing the loop
            return {
                "tool_executed": False,
                "error": str(e),
                "traceback": traceback.format_exc()
            }

    def format_result(self, result: Any) -> dict:
        """Format the result with metadata."""
        return {
            "tool_executed": True,
            "result": result,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")
        }

### Agent language: GAME ↔ prompt

- `construct_prompt`: goals → system message, memory → chat messages, actions → tool schemas.
- `parse_response`: the LLM output → `{"tool": ..., "args": ...}`; plain text is treated as a final answer.
- Tool results are sent back to the LLM as `user` messages.

In [6]:
class AgentLanguage:
    """Interface: how GAME objects become a prompt, and how the reply becomes an action."""

    def __init__(self):
        pass

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        raise NotImplementedError("Subclasses must implement this method")

    def parse_response(self, response: str) -> dict:
        raise NotImplementedError("Subclasses must implement this method")


class AgentFunctionCallingActionLanguage(AgentLanguage):
    """Language for function calling: goals -> system message, actions -> tool schemas."""

    def __init__(self):
        super().__init__()

    def format_goals(self, goals: List[Goal]) -> List:
        # All goals -> one system message: "name:\n----\ndescription\n----"
        sep = "\n-------------------\n"
        goal_instructions = "\n\n".join([f"{goal.name}:{sep}{goal.description}{sep}" for goal in goals])
        return [
            {"role": "system", "content": goal_instructions}
        ]

    def format_memory(self, memory: Memory) -> List:
        """Map memory items to chat messages."""
        # assistant decisions -> role "assistant"
        # environment results -> role "user"   (changed for MiniCPM, see above)
        # user messages       -> role "user"
        items = memory.get_memories()
        mapped_items = []
        for item in items:

            content = item.get("content", None)
            if not content:
                content = json.dumps(item, indent=4)

            if item["type"] == "assistant":
                mapped_items.append({"role": "assistant", "content": content})
            elif item["type"] == "environment":
                mapped_items.append({"role": "user", "content": content})
            else:
                mapped_items.append({"role": "user", "content": content})

        return mapped_items

    def format_actions(self, actions: List[Action]) -> [List, List]:
        """Actions -> tool schemas in the standard function-calling format."""
        tools = [
            {
                "type": "function",
                "function": {
                    "name": action.name,
                    # Include up to 1024 characters of the description
                    "description": action.description[:1024],
                    "parameters": action.parameters,
                },
            } for action in actions
        ]

        return tools

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        # messages = goals (system) + memory (conversation); tools = actions
        prompt = []
        prompt += self.format_goals(goals)
        prompt += self.format_memory(memory)

        tools = self.format_actions(actions)

        return Prompt(messages=prompt, tools=tools)

    def adapt_prompt_after_parsing_error(self,
                                         prompt: Prompt,
                                         response: str,
                                         traceback: str,
                                         error: Any,
                                         retries_left: int) -> Prompt:
        # Hook for later: repair the prompt after a parsing error (unused for now)
        return prompt

    def parse_response(self, response: str) -> dict:
        """Parse the LLM response: a JSON tool call, or plain text -> terminate."""
        try:
            return json.loads(response)

        except Exception as e:
            # Not a tool call: treat the text as the final answer
            return {
                "tool": "terminate",
                "args": {"message": response}
            }

### Agent: the loop

`agent.run(task)` runs the six steps above:

| Step | Method |
|---|---|
| 1. Construct prompt | `construct_prompt` |
| 2. Generate response | `prompt_llm_for_action` → `generate_response` |
| 3. Parse | `get_action` → invocation + `Action` |
| 4. Execute | `environment.execute_action` |
| 5. Update memory | `update_memory` |
| 6. Terminate? | `should_terminate` (terminal action) or `max_iterations` |

It prints every decision and result, and returns the final `Memory`.

In [7]:
class Agent:
    def __init__(self,
                 goals: List[Goal],
                 agent_language: AgentLanguage,
                 action_registry: ActionRegistry,
                 generate_response: Callable[[Prompt], str],
                 environment: Environment):
        """
        Initialize an agent with its core GAME components
        """
        self.goals = goals                          # G
        self.generate_response = generate_response  # the LLM
        self.agent_language = agent_language        # GAME <-> prompt translator
        self.actions = action_registry              # A
        self.environment = environment              # E   (M is passed to run())

    def construct_prompt(self, goals: List[Goal], memory: Memory, actions: ActionRegistry) -> Prompt:
        """Build prompt with memory context"""
        return self.agent_language.construct_prompt(
            actions=actions.get_actions(),
            environment=self.environment,
            goals=goals,
            memory=memory
        )

    def get_action(self, response):
        # Reply -> {"tool": ..., "args": ...} -> the registered Action object
        invocation = self.agent_language.parse_response(response)
        action = self.actions.get_action(invocation["tool"])
        return action, invocation

    def should_terminate(self, response: str) -> bool:
        action_def, _ = self.get_action(response)
        return action_def.terminal

    def set_current_task(self, memory: Memory, task: str):
        memory.add_memory({"type": "user", "content": task})

    def update_memory(self, memory: Memory, response: str, result: dict):
        """
        Update memory with the agent's decision and the environment's response.
        """
        new_memories = [
            {"type": "assistant", "content": response},
            {"type": "environment", "content": json.dumps(result)}
        ]
        for m in new_memories:
            memory.add_memory(m)

    def prompt_llm_for_action(self, full_prompt: Prompt) -> str:
        response = self.generate_response(full_prompt)
        return response

    def run(self, user_input: str, memory=None, max_iterations: int = 50) -> Memory:
        """
        Execute the GAME loop for this agent with a maximum iteration limit.
        """
        memory = memory or Memory()
        self.set_current_task(memory, user_input)

        for _ in range(max_iterations):
            # 1. Construct a prompt that includes the Goals, Actions, and the current Memory
            prompt = self.construct_prompt(self.goals, memory, self.actions)

            # 2. Generate a response from the agent
            print("Agent thinking...")
            response = self.prompt_llm_for_action(prompt)
            print(f"Agent Decision: {response}")

            # 3. Determine which action the agent wants to execute
            action, invocation = self.get_action(response)

            # 4. Execute the action in the environment
            result = self.environment.execute_action(action, invocation["args"])
            print(f"Action Result: {result}")

            # 5. Update the agent's memory with information about what happened
            self.update_memory(memory, response, result)

            # 6. Check if the agent has decided to terminate
            if self.should_terminate(response):
                break

        return memory

### `SafeAgent`

If the LLM writes a tool name that does not exist, the base `Agent` would crash. `SafeAgent` (a subclass) replaces the missing action with one that returns an error message, so the agent can correct itself and the loop never crashes.

In [8]:
class SafeAgent(Agent):
    """Agent that never crashes on an unknown tool name or a malformed invocation."""

    def get_action(self, response):
        invocation = self.agent_language.parse_response(response)

        # Malformed invocation (e.g. valid JSON without "tool") -> treat the text as the final answer
        if not isinstance(invocation, dict) or "tool" not in invocation:
            invocation = {"tool": "terminate", "args": {"message": str(response)}}
        invocation.setdefault("args", {})

        action = self.actions.get_action(invocation["tool"])
        if action is None:
            # Unknown tool -> an action whose result tells the agent what went wrong
            name = invocation["tool"]
            available = [a.name for a in self.actions.get_actions()]
            action = Action(
                name="unknown_tool",
                function=lambda **_: f"Error: unknown tool '{name}'. Available tools: {available}",
                description="", parameters={}, terminal=False,
            )
        return action, invocation

## Analysis tools

| Tool | Does | Returns |
|---|---|---|
| `list_files()` | lists `lab_data/` | file names |
| `inspect_file(file_name)` | reads one file | columns, valid / non-numeric rows, duration, max amplitude |
| `measure_period(file_name)` | upward zero crossings of the smoothed angle | period ± error, number of oscillations, initial amplitude |
| `fit_g(lengths_m, periods_s)` | least-squares fit of $T^2 = \frac{4\pi^2}{g}L$ through the origin | *g* ± error |

Design rules: tools return **short text** (it goes into the LLM's memory), report problems as **error messages**, and only **compute**. Deciding which files to trust and converting units is the agent's job. `fit_log` records every fit, to check the agent afterwards.

In [9]:
DATA_DIR = "lab_data"
fit_log = []

def _load(file_name):
    path = os.path.join(DATA_DIR, os.path.basename(file_name))
    with open(path) as f:
        header = f.readline().strip().split(",")
        t, a, bad = [], [], 0
        for line in f:
            parts = line.strip().split(",")
            try:
                ti, ai = float(parts[0]), float(parts[1])
            except (ValueError, IndexError):
                bad += 1; continue
            t.append(ti); a.append(ai)
    return header, np.array(t), np.array(a), bad

def list_files():
    """List the measurement files in the lab data folder."""
    return sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv"))

def inspect_file(file_name):
    """Summary of one file: columns, number of rows, duration, amplitude, bad values."""
    try:
        header, t, a, bad = _load(file_name)
    except FileNotFoundError:
        return f"Error: file '{file_name}' not found. Use list_files()."
    if t.size == 0:
        return f"{file_name}: no numeric data."
    return (f"{file_name}: columns {header}, {t.size} valid rows, {bad} non-numeric rows, "
            f"duration {t[-1] - t[0]:.1f} s, max |angle| {np.max(np.abs(a)):.1f} deg")

def measure_period(file_name):
    """Measure the oscillation period from the upward zero crossings of the (smoothed) angle."""
    try:
        header, t, a, bad = _load(file_name)
    except FileNotFoundError:
        return f"Error: file '{file_name}' not found. Use list_files()."
    if t.size < 50:
        return f"Error: {file_name} has too few valid data points ({t.size}) to measure a period."
    smooth = np.convolve(a, np.ones(5) / 5, mode="same")
    s = smooth[2:-2]; ts = t[2:-2]
    idx = np.where((s[:-1] < 0) & (s[1:] >= 0))[0]
    crossings = ts[idx] - s[idx] * (ts[idx + 1] - ts[idx]) / (s[idx + 1] - s[idx])   # linear interpolation
    periods = np.diff(crossings)
    if periods.size >= 2:
        periods = periods[periods > 0.5 * np.median(periods)]                       # drop noise-induced crossings
    if periods.size < 3:
        return f"Error: {file_name} is too short or too noisy: fewer than 3 full oscillations."
    T, dT = periods.mean(), periods.std(ddof=1) / np.sqrt(periods.size)
    amp0 = np.max(np.abs(a[t < T]))
    warn = f", {bad} non-numeric rows skipped" if bad else ""
    return f"{file_name}: period = {T:.4f} ± {dT:.4f} s from {periods.size} oscillations, initial amplitude {amp0:.1f} deg{warn}"

def fit_g(lengths_m, periods_s):
    """Fit T^2 = (4 pi^2 / g) * L through the origin and return g with its uncertainty."""
    L = np.asarray(lengths_m, dtype=float); T = np.asarray(periods_s, dtype=float)
    if L.size != T.size or L.size < 2:
        return "Error: give at least two lengths and the same number of periods."
    y = T ** 2
    slope = np.sum(L * y) / np.sum(L * L)
    resid = y - slope * L
    dslope = np.sqrt(np.sum(resid ** 2) / (L.size - 1) / np.sum(L * L))
    g = 4 * np.pi ** 2 / slope; dg = g * dslope / slope
    fit_log.append({"lengths_m": L.tolist(), "periods_s": T.tolist(), "g": g, "dg": dg})
    return f"g = {g:.3f} ± {dg:.3f} m/s^2 from {L.size} runs (fit of T^2 vs L through the origin)"

### Test the tools

Before giving tools to an agent, check them by hand. This is the analysis a careful physicist would do, and what each mistake would cost:

In [10]:
for f in list_files():
    print(measure_period(f))

periods = [float(measure_period(f).split("= ")[1].split(" ")[0]) for f in list_files()[:5]]
print("\nGood runs only (01-04, 75 cm -> 0.75 m):", fit_g([0.25, 0.50, 0.75, 1.00], periods[:4]))
print("Including the 50° run:                   ", fit_g([0.25, 0.50, 0.75, 1.00, 1.25], periods))
print("Reading '75cm' as 75 m:                  ", fit_g([0.25, 0.50, 75, 1.00], periods[:4]))
fit_log.clear()

run01_L=0.25m.csv: period = 0.9420 ± 0.0006 s from 31 oscillations, initial amplitude 8.1 deg
run02_L=0.50m.csv: period = 1.3318 ± 0.0013 s from 21 oscillations, initial amplitude 7.1 deg
run03_L=75cm.csv: period = 1.6320 ± 0.0018 s from 17 oscillations, initial amplitude 8.1 deg
run04_L=1.00m.csv: period = 1.8834 ± 0.0025 s from 15 oscillations, initial amplitude 6.2 deg
run05_L=1.25m.csv: period = 2.1551 ± 0.0060 s from 13 oscillations, initial amplitude 50.4 deg
Error: run06_L=1.50m.csv is too short or too noisy: fewer than 3 full oscillations.

Good runs only (01-04, 75 cm -> 0.75 m): g = 11.125 ± 0.003 m/s^2 from 4 runs (fit of T^2 vs L through the origin)
Including the 50° run:                    g = 10.892 ± 0.125 m/s^2 from 5 runs (fit of T^2 vs L through the origin)
Reading '75cm' as 75 m:                   g = 1086.615 ± 926.086 m/s^2 from 4 runs (fit of T^2 vs L through the origin)


## Step 3 · Configure and run the agent

- **Goals** give the *procedure* and the *physics* (the formula is valid only for small amplitudes), **not the answers**: the agent still has to find which files break the rules.
- **Actions** register the tools with their JSON schemas; `terminate` is the only terminal action.
- The **agent** = goals + actions + LLM + environment.

In [11]:
goals = [
    Goal(priority=1, name="Role",
         description="You are a lab assistant analysing pendulum measurements stored in files. "
                     "Use only the data and the tools: never guess values from memory and never do calculations in your head."),
    Goal(priority=2, name="Method",
         description="1. List the files. "
                     "2. For each file, measure the period (inspect the file if something looks wrong). "
                     "3. Read the pendulum length from the file name and convert it to metres. "
                     "4. Exclude runs that are corrupted or too short, and runs whose amplitude is above about 15 degrees, "
                     "because T^2 = (4 pi^2 / g) L is only valid for small oscillations. "
                     "5. Fit g with fit_g using the valid runs only. Use one tool per response."),
    Goal(priority=3, name="Terminate",
         description="Call terminate with: the value of g with its uncertainty, the runs used, "
                     "and every excluded run with the reason."),
]

action_registry = ActionRegistry()
action_registry.register(Action(
    name="list_files", function=list_files,
    description="Lists the measurement files in the lab data folder.",
    parameters={"type": "object", "properties": {}, "required": []},
))
action_registry.register(Action(
    name="inspect_file", function=inspect_file,
    description="Summary of one measurement file: columns, valid and non-numeric rows, duration, max amplitude.",
    parameters={"type": "object",
                "properties": {"file_name": {"type": "string", "description": "A name returned by list_files."}},
                "required": ["file_name"]},
))
action_registry.register(Action(
    name="measure_period", function=measure_period,
    description="Measures the oscillation period of one file, with its uncertainty and the initial amplitude in degrees.",
    parameters={"type": "object",
                "properties": {"file_name": {"type": "string", "description": "A name returned by list_files."}},
                "required": ["file_name"]},
))
action_registry.register(Action(
    name="fit_g", function=fit_g,
    description="Fits T^2 = (4 pi^2 / g) L and returns g. Lengths must be in metres, in the same order as the periods.",
    parameters={"type": "object",
                "properties": {
                    "lengths_m": {"type": "array", "items": {"type": "number"}, "description": "Pendulum lengths in metres."},
                    "periods_s": {"type": "array", "items": {"type": "number"}, "description": "Measured periods in seconds."}},
                "required": ["lengths_m", "periods_s"]},
))
action_registry.register(Action(
    name="terminate", function=lambda message: message,
    description="Ends the task and reports the result to the user.",
    parameters={"type": "object",
                "properties": {"message": {"type": "string", "description": "g ± error, runs used, runs excluded and why."}},
                "required": ["message"]},
    terminal=True,
))

agent = SafeAgent(goals=goals, agent_language=AgentFunctionCallingActionLanguage(),
                  action_registry=action_registry, generate_response=generate_response,
                  environment=Environment())

### Run

Watch the `Agent Decision` lines: which files it inspects, what it does with the corrupted file, which lengths and periods it passes to `fit_g`.

In [12]:
fit_log.clear()
task = "Determine the gravitational acceleration g of this planet from the pendulum measurements in the lab data folder."
final_memory = agent.run(task, max_iterations=30)

Agent thinking...
Agent Decision: {"tool": "list_files", "args": {}}
Action Result: {'tool_executed': True, 'result': ['run01_L=0.25m.csv', 'run02_L=0.50m.csv', 'run03_L=75cm.csv', 'run04_L=1.00m.csv', 'run05_L=1.25m.csv', 'run06_L=1.50m.csv'], 'timestamp': '2026-10-19T15:46:19+0000'}
Agent thinking...
Agent Decision: {"tool": "inspect_file", "args": {"file_name": "run01_L=0.25m.csv"}}
Action Result: {'tool_executed': True, 'result': "run01_L=0.25m.csv: columns ['time_s', 'angle_deg'], 1500 valid rows, 0 non-numeric rows, duration 30.0 s, max |angle| 8.1 deg", 'timestamp': '2026-10-19T15:46:22+0000'}
Agent thinking...
Agent Decision: {"tool": "inspect_file", "args": {"file_name": "run02_L=0.50m.csv"}}
Action Result: {'tool_executed': True, 'result': "run02_L=0.50m.csv: columns ['time_s', 'angle_deg'], 1500 valid rows, 0 non-numeric rows, duration 30.0 s, max |angle| 7.1 deg", 'timestamp': '2026-10-19T15:46:26+0000'}
Agent thinking...
Agent Decision: {"tool": "inspect_file", "args": {"f

## Baseline: the same LLM without tools

Same question, no tools, no access to the files. Without data it can only refuse or guess (e.g. Earth's 9.81 m/s²).

In [13]:
baseline = minicpm_chat([{"role": "user", "content": task + " Give a number."}], max_new_tokens=200)
print(baseline)

Apologies, I'm unable to access or analyze specific lab data files. I can only provide information based on general knowledge.


## Reveal the true *g* and score the agent

Note: even the 6–8° runs are not perfectly small-angle ($T \approx T_0(1 + \theta_0^2/16)$), so a correct analysis still comes out ~0.2% low. The ± from `fit_g` is only the statistical error, not this systematic bias.

In [14]:
print(f"True g (hidden):  {_G_TRUE:.3f} m/s^2")
if fit_log:
    last = fit_log[-1]
    err = 100 * abs(last["g"] - _G_TRUE) / _G_TRUE
    print(f"Agent's last fit: {last['g']:.3f} ± {last['dg']:.3f} m/s^2  ({err:.2f}% off)")
    print(f"  lengths used [m]: {last['lengths_m']}")
    print(f"  periods used [s]: {[round(p, 4) for p in last['periods_s']]}")
    print("  checks: 75 cm converted ->", 0.75 in last["lengths_m"],
          "| large-amplitude run excluded ->", 1.25 not in last["lengths_m"],
          "| corrupted run excluded ->", 1.5 not in last["lengths_m"])
else:
    print("The agent never called fit_g.")
print(f"Iterations used: {(len(final_memory.get_memories()) - 1) // 2}")

True g (hidden):  11.136 m/s^2
Agent's last fit: 11.125 ± 0.003 m/s^2  (0.09% off)
  lengths used [m]: [0.25, 0.5, 0.75, 1.0]
  periods used [s]: [0.942, 1.3318, 1.632, 1.8834]
  checks: 75 cm converted -> True | large-amplitude run excluded -> True | corrupted run excluded -> True
Iterations used: 15


## Results (saved run, Colab T4 GPU)

| | Value |
|---|---|
| True *g* (hidden) | **11.136** m/s² |
| Agent | **11.125 ± 0.003** m/s² (0.09% off) |
| LLM without tools | *"I'm unable to access or analyze specific lab data files."* |
| Iterations | 15 (≈ 4 s each) |

**What the agent did**
1. Listed the files, then **inspected all six** before measuring anything.
2. Measured every period; `measure_period` reported that `run06` has fewer than 3 oscillations.
3. **Converted 75 cm → 0.75 m** from the file name.
4. **Excluded** `run05` (*"amplitude 50.4° > 15°"*) and `run06` (*"too short/noisy"*).
5. Fitted *g* with the four valid runs and reported the excluded files with their reasons.

**Why it matters**
- The answer existed **only in the data**: the LLM alone could not answer; with tools and a loop, the same small model solved the task.
- The data problems were caught by **looking at tool results**, not by a fixed script.
- Each mistake would have been visible: including the 50° run gives ≈ 10.9 m/s² (−2%), reading 75 cm as 75 m gives nonsense.
- The remaining 0.1% gap is physics, not the agent: even 6–8° oscillations are slightly anharmonic, so *g* comes out a little low.